# PCA sample-size validation

Load the latest completed validation run and visualize the convergence tables. This notebook only reads existing CSV files; it does not recompute PCA.

In [11]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists() and (path / 'src/dalg').exists():
            return path
    raise RuntimeError(f'Could not find repository root from {start}')


REPO = find_repo_root()
OUTPUT_ROOT = REPO / 'dalg-cache/pile_gemma2b_models/pca_size_validation'
override = os.environ.get('PCA_VALIDATION_RUN_DIR')
if override:
    RUN_DIR = Path(override).expanduser().resolve()
else:
    completed = []
    for manifest_path in OUTPUT_ROOT.glob('*/manifest.json'):
        manifest_candidate = json.loads(manifest_path.read_text())
        if manifest_candidate.get('status') == 'complete':
            completed.append(manifest_path.parent)
    if not completed:
        raise FileNotFoundError(f'No completed validation runs under {OUTPUT_ROOT}')
    RUN_DIR = max(completed, key=lambda path: (path / 'manifest.json').stat().st_mtime_ns)

manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
summary = pd.read_csv(RUN_DIR / 'convergence_summary.csv')
metrics = pd.read_csv(RUN_DIR / 'per_cluster_metrics.csv')
selected = pd.read_csv(RUN_DIR / 'selected_clusters.csv')

print(f'Run: {RUN_DIR}')
print(f"Clusters: {len(selected)} | caps: {manifest['sample_sizes']} | top PCs: {manifest['top_pcs']}")

Run: /orfeo/cephfs/home/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/pile_gemma2b_models/pca_size_validation/layer05_K1000_q10_c128_seed0
Clusters: 128 | caps: [2000, 5000, 10000, 20000, 40000] | top PCs: 100


## What the quantities mean

The **actual estimates** table reports the median geometry across the 128 selected clusters:

- **Intrinsic dimension (90%)**: number of PCs needed to explain 90% of within-cluster variance.
- **Participation ratio**: effective number of variance-carrying directions, `(Σλ)² / Σλ²`. Larger means variance is spread across more directions.
- **Corrected isotropy**: participation ratio normalized by the finite-sample isotropic baseline. `1` is spherical; values near `0` are strongly anisotropic.

The **convergence errors** table compares every cap with the *same cluster computed from 20k samples*, then takes the median across clusters:

- **|Δ intrinsic dimension|**, **participation-ratio error**, and **|Δ corrected isotropy|**: lower is better; `0` means agreement with 20k.
- **Top-100 PC overlap**: mean squared cosine between the two 100-dimensional PC subspaces; higher is better and `1` means identical.
- **Median PC angle**: principal-angle discrepancy in degrees; lower is better and `0°` means identical.

The 20k rows are the reference compared with itself, so their errors are exactly zero, overlap is one, and angle is zero. These convergence diagnostics test sample sufficiency within each partition; they are not themselves a KMeans-vs-MFA geometry comparison.

In [12]:
actual = (
    metrics.groupby(['partition', 'sample_cap'], as_index=False)
    [['intrinsic_dim', 'participation_ratio', 'sample_corrected_isotropy']]
    .median()
    .rename(columns={
        'intrinsic_dim': 'median intrinsic dimension (90%)',
        'participation_ratio': 'median participation ratio',
        'sample_corrected_isotropy': 'median corrected isotropy',
    })
)
print('Actual estimated geometry')
display(actual.style.format({
    'median intrinsic dimension (90%)': '{:.1f}',
    'median participation ratio': '{:.2f}',
    'median corrected isotropy': '{:.4f}',
}))

vs_reference = summary.query("comparison == 'largest_cap'").copy()
summary_columns = [
    'partition', 'sample_cap', 'reference_cap',
    'intrinsic_dim_abs_delta_median',
    'participation_ratio_abs_relative_error_median',
    'isotropy_abs_delta_median',
    'pc_mean_cos2_median',
    'pc_median_angle_deg_median',
]
convergence_table = (
    vs_reference[summary_columns]
    .sort_values(['partition', 'sample_cap'])
    .rename(columns={
        'intrinsic_dim_abs_delta_median': 'median |Δ intrinsic dimension|',
        'participation_ratio_abs_relative_error_median': 'median participation-ratio error',
        'isotropy_abs_delta_median': 'median |Δ corrected isotropy|',
        'pc_mean_cos2_median': 'median top-100 PC overlap',
        'pc_median_angle_deg_median': 'median PC angle',
    })
)
print('Convergence errors relative to 20k')
display(
    convergence_table
    .style.format({
        'median participation-ratio error': '{:.2%}',
        'median |Δ corrected isotropy|': '{:.4f}',
        'median top-100 PC overlap': '{:.3f}',
        'median PC angle': '{:.2f}°',
    })
)

Actual estimated geometry


,partition,sample_cap,median intrinsic dimension (90%),median participation ratio,median corrected isotropy
0,kmeans,2000,239.5,34.07,0.0327
1,kmeans,5000,286.0,34.53,0.0231
2,kmeans,10000,304.0,34.55,0.0198
3,kmeans,20000,315.0,34.57,0.0181
4,kmeans,40000,321.5,34.65,0.0173
5,mfa_responsibility,2000,212.0,23.69,0.0225
6,mfa_responsibility,5000,250.0,23.60,0.0156
7,mfa_responsibility,10000,266.5,23.79,0.0134
8,mfa_responsibility,20000,276.5,23.88,0.0123
9,mfa_responsibility,40000,281.0,24.04,0.0118


Convergence errors relative to 20k


,partition,sample_cap,reference_cap,median |Δ intrinsic dimension|,median participation-ratio error,median |Δ corrected isotropy|,median top-100 PC overlap,median PC angle
0,kmeans,2000,40000,78.000000,2.51%,0.0154,0.861,12.82°
1,kmeans,5000,40000,34.500000,1.34%,0.0055,0.930,7.48°
2,kmeans,10000,40000,16.000000,0.80%,0.0023,0.964,4.59°
3,kmeans,20000,40000,6.000000,0.36%,0.0008,0.985,1.82°
4,kmeans,40000,40000,0.000000,0.00%,0.0000,1.000,0.00°
9,mfa_responsibility,2000,40000,66.000000,1.99%,0.0107,0.849,13.39°
10,mfa_responsibility,5000,40000,29.000000,1.00%,0.0040,0.925,7.86°
11,mfa_responsibility,10000,40000,13.000000,0.54%,0.0017,0.961,4.89°
12,mfa_responsibility,20000,40000,4.000000,0.30%,0.0006,0.984,1.99°
13,mfa_responsibility,40000,40000,0.000000,0.00%,0.0000,1.000,0.00°


In [13]:
colors = {'kmeans': '#2a78d6', 'mfa_responsibility': '#1baf7a'}
panels = [
    ('intrinsic_dim_abs_delta_median', 'Median |Δ intrinsic dimension|'),
    ('participation_ratio_abs_relative_error_median', 'Median participation-ratio error'),
    ('isotropy_abs_delta_median', 'Median |Δ corrected isotropy|'),
    ('pc_mean_cos2_median', 'Median top-100 PC overlap'),
]
for column, title in panels:
    fig = px.line(
        vs_reference.sort_values('sample_cap'),
        x='sample_cap', y=column, color='partition', markers=True,
        color_discrete_map=colors, template='plotly_white', title=title,
    )
    fig.update_traces(line_width=3, marker_size=10)
    fig.update_layout(
        height=500, xaxis_title='sample cap', yaxis_title=title,
        font=dict(size=14), legend_title_text='',
        margin=dict(l=70, r=30, t=65, b=60),
    )
    if column == 'participation_ratio_abs_relative_error_median':
        fig.update_yaxes(tickformat='.1%')
    if column == 'pc_mean_cos2_median':
        fig.update_yaxes(range=[0.8, 1.0])
    fig.show()

## Per-cluster distributions

In [14]:
plot_metrics = metrics.assign(sample_cap=metrics['sample_cap'].astype(str))
fig = px.box(
    plot_metrics, x='sample_cap', y='sample_corrected_isotropy', color='partition',
    color_discrete_map=colors, points=False, template='plotly_white',
    title='Sample-corrected isotropy across clusters',
)
fig.update_layout(xaxis_title='sample cap', yaxis_title='corrected isotropy')
fig.show()

fig = px.box(
    plot_metrics, x='sample_cap', y='intrinsic_dim', color='partition',
    color_discrete_map=colors, points=False, template='plotly_white',
    title='Intrinsic dimension across clusters',
)
fig.update_layout(xaxis_title='sample cap', yaxis_title='intrinsic dimension')
fig.show()

## Raw tables

In [15]:
display(selected)
display(metrics.sort_values(['partition', 'cluster', 'sample_cap']).head(40))

,cluster,kmeans_size,mfa_responsibility_size,joint_min_size
0,6,42420,46901,42420
1,17,46947,52322,46947
2,36,56547,66507,56547
3,39,51537,51919,51537
4,54,44626,61430,44626
...,...,...,...,...
123,969,71226,45750,45750
124,977,131160,53168,53168
125,978,50468,54563,50468
126,987,80872,59652,59652


,partition,cluster,sample_cap,intrinsic_dim,participation_ratio,sample_corrected_isotropy
0,kmeans,6,2000,183,33.986054,0.032648
128,kmeans,6,5000,214,34.703061,0.023218
256,kmeans,6,10000,228,34.626336,0.019795
384,kmeans,6,20000,233,34.463630,0.018023
512,kmeans,6,40000,235,34.397591,0.017152
1,kmeans,17,2000,202,28.201937,0.026923
129,kmeans,17,5000,238,29.084112,0.019347
257,kmeans,17,10000,251,29.368979,0.016700
385,kmeans,17,20000,261,29.431243,0.015313
513,kmeans,17,40000,265,29.500693,0.014637
